Median split on correlation of FD and raw eye movement signal
For each method i
    Run method on all runs
    Get fixed effects for good and bad runs 
    Correlate the fixed effects
Whichever method has higher mean correlation across subjects is better

In [1]:
import numpy as np 
from mreyemove.analysis.glm.config import GLMConfig
from SleepMReye.sleepmreye.sleep_mrio import SleepMRIO

signal = "raw"

config = GLMConfig.from_yaml("/Users/zach/2025_RA/matthias/eyemove_and_sleep/SleepMReye/designs/original.yml")
dataset = SleepMRIO(
    bids_root="/Users/zach/2025_RA/matthias/bids_sleepmreye",
    exclude_subjects=["01", "07", "09", "12"],
    tr=2.1,
    task="sleep",
    load_signal=True,
    desc_add=signal,
)

def proportion_awake(run_df, state: str):
    return (run_df["state"] == state).sum() / len(run_df)

subjects_to_keep = []
threshold = 0.5
state = "W"
contrast = "mreyemove_W"


for subject in dataset.subjects:
    n_awake = 0
    
    for subject, run, run_df in dataset.iter_run_events(subject):
        if proportion_awake(run_df, state) > threshold:
            n_awake += 1
         # however you get this
        # print(f"Subject {subject}, Run {run}: {n_awake_trs}/427 awake TRs ({n_awake_trs/427*100:.1f}%)")
    if n_awake < 3:
      drop = "Dropping"
    else:
        drop = "Keeping"
        subjects_to_keep.append(subject)
    print(f"{drop} subject {subject}: {n_awake} awake > {threshold} runs")
print(len(subjects_to_keep))



Keeping subject 10: 3 awake > 0.5 runs
Keeping subject 22: 3 awake > 0.5 runs
Keeping subject 06: 4 awake > 0.5 runs
Dropping subject 18: 2 awake > 0.5 runs
Dropping subject 25: 1 awake > 0.5 runs
Keeping subject 17: 5 awake > 0.5 runs
Keeping subject 33: 5 awake > 0.5 runs
Dropping subject 19: 0 awake > 0.5 runs
Keeping subject 24: 3 awake > 0.5 runs
Dropping subject 16: 0 awake > 0.5 runs
Keeping subject 32: 7 awake > 0.5 runs
Keeping subject 08: 7 awake > 0.5 runs
Dropping subject 11: 1 awake > 0.5 runs
Dropping subject 23: 1 awake > 0.5 runs
Keeping subject 26: 6 awake > 0.5 runs
Keeping subject 30: 3 awake > 0.5 runs
Dropping subject 14: 0 awake > 0.5 runs
Keeping subject 29: 7 awake > 0.5 runs
Keeping subject 13: 4 awake > 0.5 runs
Keeping subject 05: 6 awake > 0.5 runs
Dropping subject 04: 1 awake > 0.5 runs
Keeping subject 20: 3 awake > 0.5 runs
Dropping subject 27: 1 awake > 0.5 runs
Dropping subject 03: 0 awake > 0.5 runs
Dropping subject 31: 0 awake > 0.5 runs
Dropping subje

In [4]:
# CALCULATE FD CORRELATION WITH EYEMOVE AND MEDIAN SPLIT RUNS PER SUBJECT

# list of good/bad runs per subject
good_runs = {}
bad_runs = {}

for subject in subjects_to_keep:
    rs = {}
    for subject, run, run_df in dataset.iter_run_events(subject=subject):
        if proportion_awake(run_df, state) < threshold:
            continue
        FD = dataset.load_confounds(subject=subject, run=run, confound_names=["framewise_displacement"]).reset_index(drop=True)["framewise_displacement"].to_numpy()
        eye_move, _ = dataset.load_eye_move(subject=subject, run=run, desc_add="raw")
        rs[run] = (np.corrcoef(FD.ravel(), eye_move.ravel())[0,1])
        
    median_r = np.median(list(rs.values()))
    good_runs[subject] = [run for run, r in rs.items() if r <= median_r]
    bad_runs[subject] = [run for run, r in rs.items() if r > median_r]
    

In [7]:
from mreyemove.analysis.glm.first_level import combine_and_save_run_effects
from mreyemove.types.glm import GLMRunData

# SEPARATELY COMPUTE FIXED EFFECTS FOR GOOD AND BAD RUNS 

methods = ["raw-conv-fd", "clean-original"]
alt_method = ["clean-original", "raw-conv-fd"]
contrast = "mreyemove_W"

# method = methods[0]
method_rs = {}

for i, method in enumerate(methods):
    print(f"{alt_method[i]} for good runs and {method} for bad runs")
    subject_rs = []
    for subject in subjects_to_keep:
        # Grab the run data
        good_data = {
            contrast: 
                [dataset.load_run_level_glm_data(subject=subject, run=run, desc_add=alt_method[i], contrasts=[contrast])[0][contrast]
            for run in good_runs[subject]]
        }
        
        bad_data = {
            contrast: 
                [dataset.load_run_level_glm_data(subject=subject, run=run, desc_add=method, contrasts=[contrast])[0][contrast]
            for run in bad_runs[subject]]
        }
        
        # Compute fixed effects
        good_fe, _ = combine_and_save_run_effects(dataset=dataset, subject=subject, contrast_result_map=good_data, dry_run=True)
        bad_fe, _ = combine_and_save_run_effects(dataset=dataset, subject=subject, contrast_result_map=bad_data, dry_run=True)

        # Correlate fixed effects
        good_fe_data = good_fe[contrast].stat.get_fdata().ravel()
        bad_fe_data = bad_fe[contrast].stat.get_fdata().ravel()
        corr = np.corrcoef(good_fe_data, bad_fe_data)[0,1]

        subject_rs.append(corr)
    method_rs[method] = subject_rs


clean-original for good runs and raw-conv-fd for bad runs
raw-conv-fd for good runs and clean-original for bad runs


In [8]:
def fisher_z(r):
    """Fisher Z-transformation"""
    return 0.5 * np.log((1 + r) / (1 - r))


# Transform before testing
z_method1 = fisher_z(np.array(method_rs[methods[0]]))
z_method2 = fisher_z(np.array(method_rs[methods[1]]))

# Now do paired t-test on z-scores
from scipy.stats import ttest_rel, ttest_1samp
t_stat, p_value = ttest_rel(z_method1, z_method2)

print(f"{methods[0]} mean r: {np.mean(method_rs[methods[0]]):.3f} (z={np.mean(z_method1):.3f})")
print(f"{methods[1]} mean r: {np.mean(method_rs[methods[1]]):.3f} (z={np.mean(z_method2):.3f})")
print(f"Paired t-test on Fisher Z: t={t_stat:.3f}, p={p_value:.4f}")

for i in range(2):
    z_method1 = fisher_z(np.array(method_rs[methods[i]]))
    t_stat, p_value = ttest_1samp(z_method1, 0)
    # print(f"1 sample t-test for method {methods[i]} on Fisher Z: t={t_stat:.3f}, p={p_value:.4f}")
    mean_r = np.mean(method_rs[methods[i]])
    print(f"{methods[i]}: mean r = {mean_r:.3f}, t={t_stat:.3f}, p={p_value:.4f}")

raw-conv-fd mean r: 0.324 (z=0.363)
clean-original mean r: 0.330 (z=0.370)
Paired t-test on Fisher Z: t=-0.344, p=0.7362
raw-conv-fd: mean r = 0.324, t=4.658, p=0.0004
clean-original: mean r = 0.330, t=4.919, p=0.0002


In [14]:
from mreyemove.analysis.glm.first_level import combine_and_save_run_effects
from mreyemove.types.glm import GLMRunData

def compute_tmap(subject, run, method, contrasts):
    # Load and compute t maps
    run_data, _  = dataset.load_run_level_glm_data(subject=subject, run=run, desc_add=method, contrasts=contrasts)
    cope_data = run_data[contrast].cope.get_fdata()
    varcope_data = run_data[contrast].varcope.get_fdata()
    
    # Calculate t-statistic: t = COPE / sqrt(VARCOPE)
    # Add small epsilon to avoid division by zero
    epsilon = 1e-10
    tstat_data = cope_data / np.sqrt(varcope_data + epsilon)
    return tstat_data.ravel()

# SEPARATELY COMPUTE FIXED EFFECTS FOR GOOD AND BAD RUNS 

methods = ["raw-conv-fd", "clean-original"]
alt_method = ["clean-original", "raw-conv-fd"]
contrast = "mreyemove_W"

# method = methods[0]
cross_corrs = {}
good_corrs = {}
bad_corrs = {}

for i, method in enumerate(methods):
    good_rs = []
    bad_rs = []
    cross_rs = []
    for subject in subjects_to_keep:
        # Grab the run data
        good_data = [compute_tmap(subject=subject, run=run, method=method, contrasts=[contrast])
            for run in good_runs[subject]]
        
        bad_data = [compute_tmap(subject=subject, run=run, method=method, contrasts=[contrast])
            for run in bad_runs[subject]]
        
        # Skip if not enough runs
        if len(good_data) < 2 or len(bad_data) < 2:
            continue
        
        # Correlate tmaps effects
        good_corr = np.corrcoef(good_data)[0,1]
        bad_corr = np.corrcoef(bad_data)[0,1]
        
        # Compute all pairwise correlations between good and bad runs
        cross_corr = []
        for good_tmap in good_data:
            for bad_tmap in bad_data:
                corr = np.corrcoef(good_tmap, bad_tmap)[0, 1]
                cross_corr.append(corr)
        
        # Average all good-bad pairwise correlations
        cross_rs.append(np.mean(cross_corr))
        good_rs.append(good_corr)
        bad_rs.append(bad_corr)
        
    good_corrs[method] = good_rs
    bad_corrs[method] = bad_rs
    cross_corrs[method] = cross_rs


In [15]:
print(f"Good runs - {methods[0]}: {np.mean(good_corrs[methods[0]]):.3f}")
print(f"Good runs - {methods[1]}: {np.mean(good_corrs[methods[1]]):.3f}")
print(f"Bad runs - {methods[0]}: {np.mean(bad_corrs[methods[0]]):.3f}")
print(f"Bad runs - {methods[1]}: {np.mean(bad_corrs[methods[1]]):.3f}")

print(f"Good-bad (run-level) - {methods[0]}: {np.mean(cross_corrs[methods[0]]):.3f}")
print(f"Good-bad (run-level) - {methods[1]}: {np.mean(cross_corrs[methods[1]]):.3f}")

Good runs - raw-conv-fd: 0.207
Good runs - clean-original: 0.202
Bad runs - raw-conv-fd: 0.213
Bad runs - clean-original: 0.239
Good-bad (run-level) - raw-conv-fd: 0.247
Good-bad (run-level) - clean-original: 0.231
